In [3]:
import re
import json
import time
import random
import hashlib
from pathlib import Path
from typing import List, Dict, Any
from urllib.request import urlopen, Request
from urllib.error import URLError, HTTPError
from urllib.parse import quote, urlsplit, urlunsplit

from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# -----------------------------
# 설정
# -----------------------------
MODEL_NAME = "gpt-4.1-mini"

API_URLS = {
    "drivers": "https://api.jolpi.ca/ergast/f1/2026/drivers.json",
    "constructors": "https://api.jolpi.ca/ergast/f1/2026/constructors.json",
    "driver_standings": "https://api.jolpi.ca/ergast/f1/2026/driverstandings.json",
    "results": "https://api.jolpi.ca/ergast/f1/2026/results.json",
    "races": "https://api.jolpi.ca/ergast/f1/2026/races.json",
    "circuits": "https://api.jolpi.ca/ergast/f1/circuits.json",
}

OUTPUT_DIR = Path("output_qa_pipeline_jolpi_6apis_with_wiki")
OUTPUT_DIR.mkdir(exist_ok=True)

RAW_API_JSON_PATH = OUTPUT_DIR / "raw_api_data.json"
DOCS_JSON_PATH = OUTPUT_DIR / "normalized_docs.json"
CHUNKS_JSON_PATH = OUTPUT_DIR / "chunks.json"
RAW_QA_PATH = OUTPUT_DIR / "raw_qa.json"
DEDUP_QA_PATH = OUTPUT_DIR / "dedup_qa.json"
FINETUNE_JSONL_PATH = OUTPUT_DIR / "f1_finetune_qa.jsonl"

TEST_MODE = False
TARGET_FINAL_QA = 4000
MAX_WIKI_DOCS = None

GENERATION_CONFIG = {
    "concept": 1,
    "summary": 1,
    "fact": 3,
    "comparison": 1,
    "misconception": 1,
}

SLEEP_SECONDS = 0.2
MAX_RETRIES = 3
MAX_PASSES = 8

CHUNK_SIZE = 1800
CHUNK_OVERLAP = 200

# 링크 본문 수집 설정
ENABLE_LINKED_WIKI_FETCH = True
MAX_WIKI_DOCS = None      # 예: 10, 20 등으로 제한 가능
WIKI_FETCH_SLEEP = 0.3
WIKI_MIN_TEXT_LEN = 300

client = OpenAI()


# -----------------------------
# 1) 기본 유틸
# -----------------------------
def normalize_url(url: str) -> str:
    parts = urlsplit(url)
    path = quote(parts.path)
    query = quote(parts.query, safe="=&")
    return urlunsplit((parts.scheme, parts.netloc, path, query, parts.fragment))


def fetch_json(url: str) -> Any:
    safe_url = normalize_url(url)
    req = Request(safe_url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(req, timeout=30) as resp:
        return json.loads(resp.read().decode("utf-8"))


def fetch_html(url: str) -> str:
    safe_url = normalize_url(url)
    req = Request(safe_url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(req, timeout=30) as resp:
        return resp.read().decode("utf-8", errors="ignore")


def save_json(path: Path, data: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def save_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def normalize_text(text: str) -> str:
    text = text.replace("\u00ad", "")
    text = text.replace("\xa0", " ")
    text = text.replace("−", "-")
    text = text.replace("–", "-")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def slugify(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"[^\w가-힣\s-]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text[:80]


def safe_get(d: Dict[str, Any], key: str, default: str = "") -> str:
    value = d.get(key, default)
    if value is None:
        return default
    return str(value)


# -----------------------------
# 2) API 수집
# -----------------------------
def collect_all_api_data() -> Dict[str, Any]:
    data = {}
    for key, url in API_URLS.items():
        print(f"[API 수집] {key} -> {url}")
        data[key] = fetch_json(url)
        time.sleep(0.2)
    return data


# -----------------------------
# 3) 위키 본문 추출
# -----------------------------
def extract_wikipedia_main_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")

    # 불필요 요소 제거
    for tag in soup.select(
        "style, script, sup.reference, table.infobox, table.sidebar, "
        "div.navbox, div.reflist, ol.references, div.mw-references-wrap, "
        "span.mw-editsection, div.hatnote, div.thumb, table.metadata"
    ):
        tag.decompose()

    root = soup.select_one("div.mw-parser-output")
    if root is None:
        root = soup.select_one("main")
    if root is None:
        root = soup.body

    if root is None:
        return ""

    blocks = []
    for node in root.find_all(["p", "h2", "h3", "ul"], recursive=True):
        text = node.get_text(" ", strip=True)
        text = re.sub(r"\[[0-9]+\]", "", text).strip()

        if not text:
            continue

        if len(text) < 20 and node.name in {"p", "ul"}:
            continue

        if text.lower() in {
            "contents", "see also", "references", "external links", "notes"
        }:
            continue

        blocks.append(text)

    text = "\n\n".join(blocks)
    text = normalize_text(text)
    return text


def fetch_wikipedia_doc(url: str, circuit_name: str, circuit_id: str) -> Dict[str, Any] | None:
    try:
        html = fetch_html(url)
        main_text = extract_wikipedia_main_text(html)

        if len(main_text) < WIKI_MIN_TEXT_LEN:
            return None

        return {
            "doc_id": f"wiki_circuit_{circuit_id}",
            "category": "circuits_wiki",
            "title": f"{circuit_name} (Wikipedia)",
            "content": main_text,
            "metadata": {
                "url": url,
                "linked_from": f"circuit_{circuit_id}"
            },
            "source_type": "linked_wiki"
        }

    except (HTTPError, URLError, TimeoutError, ValueError) as e:
        print(f"[위키 수집 실패] {url} -> {e}")
        return None
    except Exception as e:
        print(f"[예상치 못한 위키 수집 실패] {url} -> {e}")
        return None


# -----------------------------
# 4) 문서 정규화
# -----------------------------
def build_driver_docs(api_data: Dict[str, Any]) -> List[Dict[str, Any]]:
    drivers = api_data["drivers"]["MRData"]["DriverTable"]["Drivers"]
    docs = []

    for row in drivers:
        lines = [
            f"[Driver ID] {safe_get(row, 'driverId')}",
            f"[Full Name] {safe_get(row, 'givenName')} {safe_get(row, 'familyName')}".strip(),
        ]
        if row.get("permanentNumber"):
            lines.append(f"[Permanent Number] {row['permanentNumber']}")
        if row.get("code"):
            lines.append(f"[Code] {row['code']}")
        if row.get("dateOfBirth"):
            lines.append(f"[Date of Birth] {row['dateOfBirth']}")
        if row.get("nationality"):
            lines.append(f"[Nationality] {row['nationality']}")

        docs.append({
            "doc_id": f"driver_{safe_get(row, 'driverId')}",
            "category": "drivers",
            "title": f"{safe_get(row, 'givenName')} {safe_get(row, 'familyName')}".strip(),
            "content": normalize_text("\n".join(lines)),
            "metadata": row,
            "source_type": "api"
        })

    return docs


def build_constructor_docs(api_data: Dict[str, Any]) -> List[Dict[str, Any]]:
    constructors = api_data["constructors"]["MRData"]["ConstructorTable"]["Constructors"]
    docs = []

    for row in constructors:
        lines = [
            f"[Constructor ID] {safe_get(row, 'constructorId')}",
            f"[Constructor Name] {safe_get(row, 'name')}",
        ]
        if row.get("nationality"):
            lines.append(f"[Nationality] {row['nationality']}")

        docs.append({
            "doc_id": f"constructor_{safe_get(row, 'constructorId')}",
            "category": "constructors",
            "title": safe_get(row, "name", "Constructor"),
            "content": normalize_text("\n".join(lines)),
            "metadata": row,
            "source_type": "api"
        })

    return docs


def build_driver_standing_docs(api_data: Dict[str, Any]) -> List[Dict[str, Any]]:
    standings_lists = api_data["driver_standings"]["MRData"]["StandingsTable"]["StandingsLists"]
    docs = []

    for season_block in standings_lists:
        round_no = safe_get(season_block, "round")
        season = safe_get(season_block, "season")
        standings = season_block.get("DriverStandings", [])

        for row in standings:
            driver = row.get("Driver", {})
            constructors = row.get("Constructors", [])
            constructor_names = ", ".join(c.get("name", "") for c in constructors if c.get("name"))

            lines = [
                f"[Season] {season}",
                f"[Round] {round_no}",
                f"[Position] {safe_get(row, 'position')}",
                f"[Points] {safe_get(row, 'points')}",
                f"[Wins] {safe_get(row, 'wins')}",
                f"[Driver] {safe_get(driver, 'givenName')} {safe_get(driver, 'familyName')}".strip(),
                f"[Driver ID] {safe_get(driver, 'driverId')}",
            ]
            if constructor_names:
                lines.append(f"[Constructors] {constructor_names}")

            driver_id = safe_get(driver, "driverId", f"standing_{season}_{round_no}")

            docs.append({
                "doc_id": f"driver_standing_{season}_{round_no}_{driver_id}",
                "category": "driver_standings",
                "title": f"{season} Round {round_no} Driver Standing - {safe_get(driver, 'givenName')} {safe_get(driver, 'familyName')}".strip(),
                "content": normalize_text("\n".join(lines)),
                "metadata": row,
                "source_type": "api"
            })

    return docs


def build_race_docs(api_data: Dict[str, Any]) -> List[Dict[str, Any]]:
    races = api_data["races"]["MRData"]["RaceTable"]["Races"]
    docs = []

    for row in races:
        circuit = row.get("Circuit", {})
        location = circuit.get("Location", {})

        lines = [
            f"[Season] {safe_get(row, 'season')}",
            f"[Round] {safe_get(row, 'round')}",
            f"[Race Name] {safe_get(row, 'raceName')}",
            f"[Date] {safe_get(row, 'date')}",
            f"[Circuit Name] {safe_get(circuit, 'circuitName')}",
            f"[Circuit ID] {safe_get(circuit, 'circuitId')}",
            f"[Locality] {safe_get(location, 'locality')}",
            f"[Country] {safe_get(location, 'country')}",
        ]
        if location.get("lat"):
            lines.append(f"[Latitude] {location['lat']}")
        if location.get("long"):
            lines.append(f"[Longitude] {location['long']}")

        docs.append({
            "doc_id": f"race_{safe_get(row, 'season')}_{safe_get(row, 'round')}",
            "category": "races",
            "title": safe_get(row, "raceName", "Race"),
            "content": normalize_text("\n".join(lines)),
            "metadata": row,
            "source_type": "api"
        })

    return docs


def build_circuit_docs(api_data: Dict[str, Any]) -> List[Dict[str, Any]]:
    circuits = api_data["circuits"]["MRData"]["CircuitTable"]["Circuits"]
    docs = []

    for row in circuits:
        location = row.get("Location", {})
        lines = [
            f"[Circuit ID] {safe_get(row, 'circuitId')}",
            f"[Circuit Name] {safe_get(row, 'circuitName')}",
            f"[Locality] {safe_get(location, 'locality')}",
            f"[Country] {safe_get(location, 'country')}",
        ]
        if location.get("lat"):
            lines.append(f"[Latitude] {location['lat']}")
        if location.get("long"):
            lines.append(f"[Longitude] {location['long']}")
        if row.get("url"):
            lines.append(f"[Reference URL] {row['url']}")

        docs.append({
            "doc_id": f"circuit_{safe_get(row, 'circuitId')}",
            "category": "circuits",
            "title": safe_get(row, "circuitName", "Circuit"),
            "content": normalize_text("\n".join(lines)),
            "metadata": row,
            "source_type": "api"
        })

    return docs


def build_result_docs(api_data: Dict[str, Any]) -> List[Dict[str, Any]]:
    races = api_data["results"]["MRData"]["RaceTable"]["Races"]
    docs = []

    for race in races:
        season = safe_get(race, "season")
        round_no = safe_get(race, "round")
        race_name = safe_get(race, "raceName")
        results = race.get("Results", [])

        lines = [
            f"[Season] {season}",
            f"[Round] {round_no}",
            f"[Race Name] {race_name}",
            f"[Date] {safe_get(race, 'date')}",
            "",
            "[Results]"
        ]

        for row in results:
            driver = row.get("Driver", {})
            constructor = row.get("Constructor", {})
            line = (
                f"- Position {safe_get(row, 'position')}: "
                f"{safe_get(driver, 'givenName')} {safe_get(driver, 'familyName')} "
                f"({safe_get(constructor, 'name')}) | "
                f"Grid {safe_get(row, 'grid')} | "
                f"Points {safe_get(row, 'points')} | "
                f"Status {safe_get(row, 'status')}"
            )
            lines.append(line)

        docs.append({
            "doc_id": f"race_result_{season}_{round_no}",
            "category": "results",
            "title": f"{race_name} Results",
            "content": normalize_text("\n".join(lines)),
            "metadata": race,
            "source_type": "api"
        })

    return docs


def build_linked_wiki_docs_from_circuits(api_data: Dict[str, Any]) -> List[Dict[str, Any]]:
    if not ENABLE_LINKED_WIKI_FETCH:
        return []

    circuits = api_data["circuits"]["MRData"]["CircuitTable"]["Circuits"]
    wiki_docs = []
    count = 0

    for row in circuits:
        if MAX_WIKI_DOCS is not None and count >= MAX_WIKI_DOCS:
            break

        url = row.get("url")
        if not url:
            continue

        circuit_id = safe_get(row, "circuitId")
        circuit_name = safe_get(row, "circuitName", circuit_id)

        print(f"[위키 링크 수집] {circuit_name} -> {url}")
        wiki_doc = fetch_wikipedia_doc(url, circuit_name, circuit_id)

        if wiki_doc:
            wiki_docs.append(wiki_doc)
            count += 1

        time.sleep(WIKI_FETCH_SLEEP)

    return wiki_docs


def load_all_docs_from_api(api_data: Dict[str, Any]) -> List[Dict[str, Any]]:
    docs = []
    docs.extend(build_driver_docs(api_data))
    docs.extend(build_constructor_docs(api_data))
    docs.extend(build_driver_standing_docs(api_data))
    docs.extend(build_race_docs(api_data))
    docs.extend(build_circuit_docs(api_data))
    docs.extend(build_result_docs(api_data))
    docs.extend(build_linked_wiki_docs_from_circuits(api_data))
    return docs


# -----------------------------
# 5) 청킹
# -----------------------------
def split_long_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    text = normalize_text(text)

    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + chunk_size, text_len)
        chunk = text[start:end].strip()

        if end < text_len:
            last_break = max(
                chunk.rfind("\n\n"),
                chunk.rfind("\n- "),
                chunk.rfind(". "),
                chunk.rfind("? "),
                chunk.rfind("! "),
            )
            if last_break > int(chunk_size * 0.6):
                chunk = chunk[:last_break + 1].strip()
                end = start + len(chunk)

        if chunk:
            chunks.append(chunk)

        if end >= text_len:
            break

        start = max(end - overlap, start + 1)

    return chunks


def make_chunks(docs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    chunks = []

    for doc in docs:
        parts = split_long_text(doc["content"])

        for idx, part in enumerate(parts, start=1):
            chunks.append({
                "chunk_id": f"{doc['doc_id']}__chunk_{idx}",
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "category": doc["category"],
                "content": part,
                "metadata": doc["metadata"],
                "chunk_index": idx,
                "num_chunks_in_doc": len(parts),
                "source_type": doc.get("source_type", "api")
            })

    return chunks


# -----------------------------
# 6) 프롬프트 / 생성
# -----------------------------
def build_prompt(chunk: Dict[str, Any], generation_config: Dict[str, int], pass_index: int = 1) -> str:
    question_type_desc = {
        "concept": "개념 설명형",
        "summary": "요약형",
        "fact": "사실 확인형",
        "comparison": "비교형",
        "misconception": "오해 교정형"
    }

    type_lines = []
    for qtype, n in generation_config.items():
        if n > 0:
            type_lines.append(f"- {qtype}: {n}개 ({question_type_desc[qtype]})")

    joined_types = "\n".join(type_lines)
    seed_hint = f"{chunk['chunk_id']}-{pass_index}-{random.randint(1000, 9999)}"

    prompt = f"""
너는 F1 입문자용 설명 챗봇의 파인튜닝 데이터를 만드는 역할이야.

아래 문서 일부를 읽고, 문서 내용만을 바탕으로 한국어 Q/A를 생성해라.

[문서 정보]
- 문서 ID: {chunk['doc_id']}
- 청크 ID: {chunk['chunk_id']}
- 제목: {chunk['title']}
- 카테고리: {chunk['category']}
- 소스 타입: {chunk['source_type']}
- 회차: {pass_index}
- variation_hint: {seed_hint}

[문서 내용]
{chunk['content']}

[생성 개수]
{joined_types}

[생성 규칙]
- 문서 내용만 바탕으로 작성
- 한국어로 작성
- 초보자도 이해하기 쉽게 설명
- 답변은 1~3문장
- 숫자, 연도, 순위, 포인트, 이름, 국적, 팀명, 레이스명, 서킷명은 문서 기준 그대로 유지
- 문서에 없는 사실, 이유, 배경, 목적을 추론하지 말 것
- source_type이 api이면 사실형 질문을 우선할 것
- source_type이 linked_wiki이면 요약형/개념형도 허용하되, 문서 본문에 있는 설명만 사용할 것
- 질문끼리 의미 중복이 크지 않게 작성
- 비교형은 같은 청크 안에서 비교 가능한 대상이 있을 때만 만들 것
- 오해 교정형은 왜 틀렸는지도 문서 범위 안에서 설명할 것
- 위치 정보만 있는 문서에서는 위치/국가/좌표/이름 중심 질문을 우선 만들 것
- 단순 문장 재진술만 하지 말고, 입문자 질문처럼 자연스럽게 만들 것
- 문서에 없는 커리어 해설, 역사적 평가, 추측성 설명 금지
- 문서에 없는 속성(성별, 인기도, 일반적 사용 빈도, 사회적 평가)은 질문으로 만들지 말 것
- 오해 교정형은 문서 안의 명확한 사실과 직접 충돌하는 경우에만 생성할 것

[출력 형식]
반드시 아래 JSON 배열만 출력:
[
  {{
    "question": "...",
    "answer": "..."
  }}
]
""".strip()

    return prompt


def extract_json_array(text: str) -> str:
    match = re.search(r"\[\s*{.*?}\s*\]", text, re.DOTALL)
    if not match:
        raise ValueError("JSON 배열을 찾지 못했습니다.")
    return match.group(0)


def validate_qa_items(data: Any, chunk: Dict[str, Any]) -> List[Dict[str, Any]]:
    validated = []

    if not isinstance(data, list):
        return validated

    for item in data:
        if not isinstance(item, dict):
            continue

        question = item.get("question", "")
        answer = item.get("answer", "")

        if not isinstance(question, str) or not isinstance(answer, str):
            continue

        question = question.strip()
        answer = answer.strip()

        if not question or not answer:
            continue

        validated.append({
            "question": question,
            "answer": answer,
            "doc_id": chunk["doc_id"],
            "chunk_id": chunk["chunk_id"],
            "title": chunk["title"],
            "category": chunk["category"],
            "source_type": chunk.get("source_type", "api")
        })

    return validated


def generate_qa_for_chunk(chunk: Dict[str, Any], generation_config: Dict[str, int], pass_index: int = 1) -> List[Dict[str, Any]]:
    prompt = build_prompt(chunk, generation_config, pass_index)

    for attempt in range(MAX_RETRIES):
        try:
            response = client.responses.create(
                model=MODEL_NAME,
                input=prompt
            )
            text = response.output_text.strip()

            try:
                data = json.loads(text)
            except json.JSONDecodeError:
                cleaned = extract_json_array(text)
                data = json.loads(cleaned)

            validated = validate_qa_items(data, chunk)
            if validated:
                return validated

        except Exception as e:
            print(f"[재시도 {attempt+1}/{MAX_RETRIES}] {chunk['chunk_id']} 생성 실패: {e}")
            time.sleep(1.5)

    return []


# -----------------------------
# 7) 중복 제거
# -----------------------------
def normalize_question(q: str) -> str:
    q = q.strip().lower()
    q = re.sub(r"[^\w가-힣\s]", "", q)
    q = re.sub(r"\s+", " ", q)
    return q


def normalize_answer(a: str) -> str:
    a = a.strip().lower()
    a = re.sub(r"\s+", " ", a)
    return a


def make_hash(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def deduplicate_qa(data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    deduped = []

    for item in data:
        q_norm = normalize_question(item["question"])
        a_norm = normalize_answer(item["answer"])
        key = make_hash(q_norm + "||" + a_norm)

        if key in seen:
            continue

        seen.add(key)
        deduped.append(item)

    return deduped


# -----------------------------
# 8) 파인튜닝 형식 변환
# -----------------------------
def to_finetune_messages(item: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "messages": [
            {
                "role": "user",
                "content": item["question"]
            },
            {
                "role": "assistant",
                "content": item["answer"]
            }
        ]
    }


# -----------------------------
# 9) 목표 개수까지 생성
# -----------------------------
def generate_until_target(
    chunks: List[Dict[str, Any]],
    generation_config: Dict[str, int],
    target_final_qa: int
) -> List[Dict[str, Any]]:
    raw_qa: List[Dict[str, Any]] = []
    pass_index = 1

    while pass_index <= MAX_PASSES:
        print(f"\n===== PASS {pass_index}/{MAX_PASSES} 시작 =====")
        random.shuffle(chunks)

        for idx, chunk in enumerate(chunks, start=1):
            deduped_so_far = deduplicate_qa(raw_qa)

            if len(deduped_so_far) >= target_final_qa:
                print("목표 개수 도달. 생성 종료.")
                return raw_qa

            print(
                f"  - PASS {pass_index} | ({idx}/{len(chunks)}) "
                f"{chunk['chunk_id']} | {chunk['title']} | {chunk['source_type']}"
            )

            qa_items = generate_qa_for_chunk(chunk, generation_config, pass_index=pass_index)
            raw_qa.extend(qa_items)

            if idx % 20 == 0:
                save_json(RAW_QA_PATH, raw_qa)
                dedup_snapshot = deduplicate_qa(raw_qa)
                save_json(DEDUP_QA_PATH, dedup_snapshot)
                print(
                    f"    누적 원본: {len(raw_qa)} | "
                    f"누적 중복제거: {len(dedup_snapshot)} / 목표 {target_final_qa}"
                )

            time.sleep(SLEEP_SECONDS)

        pass_index += 1

    print("최대 PASS 수에 도달했습니다.")
    return raw_qa


# -----------------------------
# 10) 실행
# -----------------------------
def main():
    print("[1/6] API 데이터 수집")
    api_data = collect_all_api_data()
    save_json(RAW_API_JSON_PATH, api_data)

    print("[2/6] 문서 정규화 + 링크 본문 수집")
    docs = load_all_docs_from_api(api_data)
    print(f"전체 문서 수: {len(docs)}")

    if TEST_MODE:
        docs = docs[:TEST_DOC_LIMIT]
        print(f"테스트 문서 수: {len(docs)}")

    save_json(DOCS_JSON_PATH, docs)

    print("[3/6] 문서 청킹")
    chunks = make_chunks(docs)
    print(f"전체 청크 수: {len(chunks)}")
    save_json(CHUNKS_JSON_PATH, chunks)

    print("\n[청크 샘플 20개]")
    for c in chunks[:20]:
        print(f"{c['chunk_id']} | {c['category']} | {c['source_type']} | {c['title']}")

    print("[4/6] 생성 설정 확인")
    per_chunk_raw = sum(GENERATION_CONFIG.values())
    print(f"청크당 요청 생성 수: {per_chunk_raw}")
    print(f"목표 최종 Q/A 수: {TARGET_FINAL_QA}")

    print("[5/6] Q/A 생성")
    raw_qa = generate_until_target(
        chunks=chunks,
        generation_config=GENERATION_CONFIG,
        target_final_qa=TARGET_FINAL_QA
    )
    save_json(RAW_QA_PATH, raw_qa)
    print(f"생성된 원본 Q/A 수: {len(raw_qa)}")

    print("[6/6] 중복 제거 및 JSONL 저장")
    deduped = deduplicate_qa(raw_qa)

    if len(deduped) > TARGET_FINAL_QA:
        deduped = deduped[:TARGET_FINAL_QA]

    save_json(DEDUP_QA_PATH, deduped)
    finetune_rows = [to_finetune_messages(item) for item in deduped]
    save_jsonl(FINETUNE_JSONL_PATH, finetune_rows)

    print(f"중복 제거 후 최종 Q/A 수: {len(deduped)}")
    print("\n완료")
    print(f"- 원본 API 데이터: {RAW_API_JSON_PATH}")
    print(f"- 정규화 문서: {DOCS_JSON_PATH}")
    print(f"- 청크 저장: {CHUNKS_JSON_PATH}")
    print(f"- 원본 Q/A: {RAW_QA_PATH}")
    print(f"- 중복 제거 Q/A: {DEDUP_QA_PATH}")
    print(f"- 파인튜닝 JSONL: {FINETUNE_JSONL_PATH}")

    if len(deduped) < TARGET_FINAL_QA:
        print(
            f"\n[주의] 목표 {TARGET_FINAL_QA}개보다 적은 {len(deduped)}개만 확보되었습니다. "
            "GENERATION_CONFIG를 늘리거나 MAX_PASSES를 올리세요."
        )


if __name__ == "__main__":
    main()

[1/6] API 데이터 수집
[API 수집] drivers -> https://api.jolpi.ca/ergast/f1/2026/drivers.json
[API 수집] constructors -> https://api.jolpi.ca/ergast/f1/2026/constructors.json
[API 수집] driver_standings -> https://api.jolpi.ca/ergast/f1/2026/driverstandings.json
[API 수집] results -> https://api.jolpi.ca/ergast/f1/2026/results.json
[API 수집] races -> https://api.jolpi.ca/ergast/f1/2026/races.json
[API 수집] circuits -> https://api.jolpi.ca/ergast/f1/circuits.json
[2/6] 문서 정규화 + 링크 본문 수집
[위키 링크 수집] Adelaide Street Circuit -> https://en.wikipedia.org/wiki/Adelaide_Street_Circuit
[위키 링크 수집] Ain Diab -> https://en.wikipedia.org/wiki/Ain-Diab_Circuit
[위키 링크 수집] Aintree -> https://en.wikipedia.org/wiki/Aintree_Motor_Racing_Circuit
[위키 링크 수집] Albert Park Grand Prix Circuit -> https://en.wikipedia.org/wiki/Albert_Park_Circuit
[위키 링크 수집] Circuit of the Americas -> https://en.wikipedia.org/wiki/Circuit_of_the_Americas
[위키 링크 수집] Scandinavian Raceway -> https://en.wikipedia.org/wiki/Anderstorp_Raceway
[위키 링크 수집] 